# Project 2 Deep Learning CSCE 636    




# Joseph DeLeonardis UIN: 820000866 April 6 2026

#Background: The LP Algorithm
The m-height of ta linear code is computer using a Linear programming (LP) algorithm. For each sample, the generator constructs a generator matrix G=[I|P| where I is an identity matrix and P is a randomly generated parity matrix. The algorithm then enumerates over all the m-subsets S of the n code coordinates and for each subset solves a linear program that maximizes the projection of a codeword onto each column direction, subject to the constraint that all other coordinates stay within [-1,1]. The m-height is the largest value found across all these LP solutions. This is computationally expensive. The class was provided a research paper that shows how it can be done with an efficient time complexity but it still posed a challenge.


#Project 1: Design Choices (Up to the third version)
Several iterations were explored to improve model performance. The first attempt used a single generalized model trained on all 96,524 samples across all 9 (n,k,m) combinations. This approach produced an overall average cost of approximately 0.77. The model struggled because it was asked to simultaneously learn 9 fundamentally different mathematical relationships between the parity matrix P and the m-height using a single set of weights, which proved to be too much for one network to capture effectively.
The second iteration addressed this by training a separate specialized model for each (n,k,m) combination. Feature engineering was also improved, expanding the input from 23 to 36 features by adding sorted column norms, row norms, and additional structural ratios. This brought the overall average cost down to approximately 0.62, confirming that the specialized approach was meaningfully better.
Three combinations continued to show poor results (9,4,5), (9,5,4), and (9,6,3)  with validation losses above 2.1. An attempt was made to improve these by generating additional synthetic training data targeting the underrepresented regions of the height distribution. However this made performance worse, pushing the overall cost up to 1.74. The new samples introduced a distribution shift that the model could not handle, so this approach was abandoned.
The third iteration kept the 9 specialized models but doubled the network architecture  wider layers and an additional hidden layer. This addressed an underfitting problem where the model was not learning the training data well enough regardless of the validation performance. The larger architecture brought the overall average cost down to approximately 0.52. The three problematic combinations still show the highest individual losses due to the mathematical complexity described in the drawbacks section below, but the larger model reduced the overall cost meaningfully compared to all previous versions.

#Last Cell Information
Per the message we got on canvas, I wrote the last cell to run the existing model without the code that would rebuild the model such that a single click can run everything.

#Drive Authorization (if needed)

Attached is the link anyone with the link has access to my CSCE 636 stuff
https://drive.google.com/drive/folders/1LQDy43SQoMe_XayfYz1dpKsNYhpVerxb?usp=sharing

#Final Updates (From project 1)
Before the submission deadline, I wanted to see if I could improve results for the three worst-performing combinations: (9,4,5), (9,5,4), and (9,6,3). These combinations are inherently challenging because m equals the code redundancy n-k, which is the largest value m can take while the height remains finite. At this saturation point the search space is maximally complex and the height values span an enormous range log2(h) from approximately 4 to 22 compared to roughly 1 to 9 for the easier combinations.

Corollary 11 gives a closed form characterization of the height exactly when m = n-k. It states that the height equals the maximum L1 norm of inverse submatrix products of the parity check matrix. Rather than expecting the model to discover this relationship from raw P matrix entries alone, these values were precomputed and added as 10 additional input features for the three hard combinations. This brought the overall average cost from 0.5273 down to 0.2389.

In [10]:
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')



TRAIN_DATA_PATH   = '/content/drive/MyDrive/CSCE_636/project_1/CSCE-636-Project-2-Train-n_k_m_P'
TRAIN_HEIGHT_PATH = '/content/drive/MyDrive/CSCE_636/project_1/CSCE-636-Project-2-Train-mHeights'
MODEL_SAVE_PATH   = '/content/drive/MyDrive/CSCE_636/project_1/best_model.pth'
NUM_EPOCHS        = 100
BATCH_SIZE        = 1024
LEARNING_RATE     = 1e-3

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
#As a sanity check make sure a GPU is selected that can handle this computation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cuda


In [12]:

with open(TRAIN_DATA_PATH, 'rb') as f:
    train_data = pickle.load(f)
with open(TRAIN_HEIGHT_PATH, 'rb') as f:
    train_heights = pickle.load(f)
print(f'Total samples: {len(train_data):,}')


# Load both LP bound files and merge
with open('/content/drive/MyDrive/CSCE_636/project_2/lp_bounds_hard_combos.pkl', 'rb') as f:
    lp_bounds = pickle.load(f)

with open('/content/drive/MyDrive/CSCE_636/project_2/lp_bounds_extra_combos.pkl', 'rb') as f:
    lp_bounds_extra = pickle.load(f)

lp_bounds.update(lp_bounds_extra)
print(f'Total LP bounds loaded: {len(lp_bounds):,}')

from itertools import combinations

def compute_c11_features(n, k, P):
    """Compute top Corollary 11 L1 norm values as features."""
    r = n - k
    H = np.hstack([P.T, np.eye(r)])
    col_indices = list(range(n))
    values = []
    for S in combinations(col_indices, r):
        S = list(S)
        H_S = H[:, S]
        if abs(np.linalg.det(H_S)) < 1e-10:
            continue
        H_S_inv = np.linalg.inv(H_S)
        H_S_bar = H[:, [j for j in col_indices if j not in S]]
        for idx in range(len(S)):
            row = H_S_inv[idx, :]
            val = np.sum(np.abs(row @ H_S_bar))
            values.append(val)
    values = sorted(values, reverse=True)
    feat = np.zeros(10)
    feat[:min(len(values), 10)] = values[:10]
    return feat


HARD_COMBOS = {(9,4,5), (9,5,4), (9,6,3)}
EXACT_MAX_COMBOS = {(9,4,5), (9,5,4), (9,6,3), (9,4,4), (9,5,3)}

def encode_sample(sample, sample_idx=None):
    n, k, m, P = sample
    col_norms = np.linalg.norm(P, axis=0)
    sort_idx  = np.argsort(-col_norms)
    P_sorted  = P[:, sort_idx]
    p_flat    = P_sorted.flatten()
    p_padded  = np.zeros(20)
    p_padded[:len(p_flat)] = p_flat
    p_padded  = p_padded / 30.0
    col_norms_padded = np.zeros(5)
    col_norms_padded[:len(col_norms)] = np.sort(col_norms)[::-1]
    col_norms_padded = col_norms_padded / 30.0
    row_norms = np.linalg.norm(P, axis=1)
    row_norms_padded = np.zeros(6)
    row_norms_padded[:len(row_norms)] = row_norms
    row_norms_padded = row_norms_padded / 30.0
    scalars = np.array([n/9.0, k/9.0, m/9.0, m/(n-k), k/n])
    combo = (int(n), int(k), int(m))
    if combo in HARD_COMBOS:
        c11_feats = compute_c11_features(n, k, P) / 10000.0
    else:
        c11_feats = np.zeros(10)
    if combo in EXACT_MAX_COMBOS and sample_idx is not None and sample_idx in lp_bounds:
        lp_feat = np.array([np.log2(lp_bounds[sample_idx]) / 25.0])
    else:
        lp_feat = np.array([0.0])
    return np.concatenate([scalars, p_padded, col_norms_padded, row_norms_padded, c11_feats, lp_feat]).astype(np.float32)



Total samples: 803,951
Total LP bounds loaded: 389,972


#Project 2 Updates Rev 1
The primary improvement in Project 2 came from analyzing the training curves of each individual (n,k,m) combination and recognizing that a single dropout rate applied uniformly across all 9 models was suboptimal. Each combination has a fundamentally different mathematical structure and therefore a different tendency to overfit or underfit. Rather than using one architecture for all combos, a per-combination dropout schedule was designed based on the observed gap between training and validation loss for each model.
The three hardest combinations (9,4,5), (9,5,4), and (9,6,3)  showed severe overfitting with validation loss spiking far above training loss each epoch. These were given significantly higher dropout rates (0.3–0.5) to force better generalization. The easier combinations which showed training and validation loss tracking closely were kept at the Project 1 dropout values of 0.2/0.1. This targeted per-combo approach brought the overall average cost from 0.2554 down to 0.2475.
An attempt was also made to improve performance through synthetic data generation targeting the underrepresented low-height region of the hard combo distributions. Histogram analysis confirmed that the bad performers have a wide Gaussian distribution centered around log₂(h) ≈ 13 compared to the more structured multimodal distributions of the easier combinations. Several versions of a data generator were developed to try to reshape this distribution and add a second peak around log₂(h) ≈ 7.5. However adding synthetic data consistently degraded model performance due to distribution shift between the generated samples and the validation set. The histograms are included to document this analysis and explain why the synthetic data approach was ultimately abandoned in favor of the dropout tuning approach.
The final overall average cost of 0.2475 represents a 68% reduction from the initial baseline of 0.77 achieved in early Project 1 iterations.

##Project 2 Rev 2
## LP-Based Feature Engineering for Hard Combos

### Motivation

During initial training, the parameter combinations (n=9, k=4, m=5) and (n=9, k=5, m=4) exhibited severe overfitting training loss converged well but validation loss remained high and unstable. Analysis of the training data revealed these two combos share a critical mathematical property: m = n-k (i.e., m equals the redundancy r), placing them at the boundary of the Singleton bound.

According to Corollary 11 in Jiang (2024) and Theorem 1 in Roth et al. (2026), when m = r the m-height of the code equals the global maximum of an LP objective over all m-subsets of the generator matrix columns. This is a fundamentally harder function to learn than the local matrix properties (column norms, row norms, P matrix values) that work well for the easier combos where m < r.

### Solution: Precomputed LP Bounds

Rather than attempting architectural fixes, we implemented the LP algorithm directly from Figure 1 of Roth et al. (2026) using scipy's HiGHS solver. For each training sample in (9,4,5) and (9,5,4), we solved the LP maximization problem and stored the result as a precomputed lookup table (lp_bounds_hard_combos.pkl). This was computed using 4 parallel CPU cores across 134,083 samples. The file exactheight.py computed the .pkl file and is also included in the submission.

The LP bound is fed to the network as a single normalized feature:

    lp_feat = np.log2(lp_bounds[sample_idx]) / 25.0

This transforms the learning problem from predicting a complex global combinatorial function from scratch to learning a small correction factor on top of a near-exact theoretical answer. For test samples not in the precomputed table, the LP bound is computed on the fly using compute_lp_bound_online.

### Results

| Combo   | Before LP feature | After LP feature |
|---------|------------------|-----------------|
| (9,4,5) | 0.47             | 0.06            |
| (9,5,4) | 0.33             | 0.14            |
| Overall | 0.19             | 0.15            |

#Project 2 Data Statisics
Shown below I am checking how the data processing needs to change with this new data set.

In [13]:
# Check Project 2 data stats
import pickle
import numpy as np

with open(TRAIN_DATA_PATH, 'rb') as f:
    p2_data = pickle.load(f)
with open(TRAIN_HEIGHT_PATH, 'rb') as f:
    p2_heights = pickle.load(f)

print(f'Total samples: {len(p2_data):,}')
log2h = np.log2(np.array(p2_heights))
print(f'log2h mean: {log2h.mean():.3f}')
print(f'log2h std:  {log2h.std():.3f}')
print(f'log2h min:  {log2h.min():.3f}')
print(f'log2h max:  {log2h.max():.3f}')

# Check P matrix scale
sample = p2_data[0]
print(f'\nFirst sample n={sample[0]}, k={sample[1]}, m={sample[2]}')
print(f'P abs mean: {np.abs(sample[3]).mean():.3f}')
print(f'P abs max:  {np.abs(sample[3]).max():.3f}')

Total samples: 803,951
log2h mean: 8.951
log2h std:  3.155
log2h min:  1.000
log2h max:  25.228

First sample n=9, k=4, m=2
P abs mean: 15.250
P abs max:  29.000


In [14]:
class MHeightDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class MHeightDNN(nn.Module):
    def __init__(self, d1=0.2, d2=0.2, d3=0.2, d4=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(47, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(d1),
            nn.Linear(512, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(d2),
            nn.Linear(1024, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(d3),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(d4),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def custom_loss(pred, true):
    return torch.mean((true - pred) ** 2)

MODEL_DIR = '/content/drive/MyDrive/CSCE_636/project_2/models'
import os
os.makedirs(MODEL_DIR, exist_ok=True)

COMBO_DROPOUT = {
    (9,4,2): (0.20, 0.20, 0.20, 0.10),
    (9,4,3): (0.20, 0.20, 0.20, 0.10),
    (9,4,4): (0.20, 0.20, 0.20, 0.10),
    (9,4,5): (0.40, 0.40, 0.40, 0.30),
    (9,5,2): (0.20, 0.20, 0.20, 0.10),
    (9,5,3): (0.20, 0.20, 0.20, 0.10),
    (9,5,4): (0.50, 0.50, 0.50, 0.40),
    (9,6,2): (0.20, 0.20, 0.20, 0.10),
    (9,6,3): (0.40, 0.40, 0.40, 0.30),
}

PARAM_COMBOS = [
    (9,4,5),(9,5,4),(9,6,3),
    (9,4,2),(9,4,3),(9,4,4),
    (9,5,2),(9,5,3),(9,6,2),
]

def model_path(n, k, m):
    return f'{MODEL_DIR}/model_n{n}_k{k}_m{m}.pth'

combo_results = {}
for (n, k, m) in PARAM_COMBOS:
    if os.path.exists(model_path(n, k, m)):
        print(f'Skipping ({n},{k},{m}) — model already exists')
        combo_results[(n,k,m)] = None
        continue

    print(f'\n{"="*50}')
    print(f'Training (n={n}, k={k}, m={m})')
    idx = [i for i, s in enumerate(train_data) if s[0]==n and s[1]==k and s[2]==m]
    combo_data    = [train_data[i] for i in idx]
    combo_heights = [train_heights[i] for i in idx]
    X = np.array([encode_sample(s, sample_idx=idx[i]) for i, s in enumerate(combo_data)])
    y = np.array([np.log2(h) for h in combo_heights], dtype=np.float32)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)
    train_loader = DataLoader(MHeightDataset(X_train, y_train), batch_size=512, shuffle=True)
    val_loader   = DataLoader(MHeightDataset(X_val, y_val), batch_size=512, shuffle=False)

    d1, d2, d3, d4 = COMBO_DROPOUT[(n, k, m)]
    model = MHeightDNN(d1=d1, d2=d2, d3=d3, d4=d4).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    best_val_loss = float('inf')
    for epoch in range(100):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = custom_loss(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                val_loss += custom_loss(model(X_batch), y_batch).item()
        val_loss /= len(val_loader)
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_path(n, k, m))
        if epoch % 20 == 0 or epoch == 99:
            print(f'  Epoch {epoch:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Best: {best_val_loss:.4f}')
    combo_results[(n,k,m)] = best_val_loss

print('\nAll done:')
for combo, loss in combo_results.items():
    if loss is not None:
        print(f'  {combo}: {loss:.4f}')
    else:
        print(f'  {combo}: skipped')


Training (n=9, k=4, m=5)
  Epoch   0 | Train: 9.3292 | Val: 1.2579 | Best: 1.2579
  Epoch  20 | Train: 0.4261 | Val: 0.6410 | Best: 0.2146
  Epoch  40 | Train: 0.3156 | Val: 0.1111 | Best: 0.1111
  Epoch  60 | Train: 0.2940 | Val: 0.2841 | Best: 0.1111
  Epoch  80 | Train: 0.2871 | Val: 0.4324 | Best: 0.1111
  Epoch  99 | Train: 0.2856 | Val: 0.4199 | Best: 0.1111

Training (n=9, k=5, m=4)
  Epoch   0 | Train: 15.0527 | Val: 1.2247 | Best: 1.2247
  Epoch  20 | Train: 0.6919 | Val: 0.3943 | Best: 0.3943
  Epoch  40 | Train: 0.5845 | Val: 0.3664 | Best: 0.3164
  Epoch  60 | Train: 0.5362 | Val: 0.7582 | Best: 0.1768
  Epoch  80 | Train: 0.5255 | Val: 0.6508 | Best: 0.1768
  Epoch  99 | Train: 0.5400 | Val: 3.2836 | Best: 0.1768

Training (n=9, k=6, m=3)
  Epoch   0 | Train: 9.9675 | Val: 0.3749 | Best: 0.3749
  Epoch  20 | Train: 0.3805 | Val: 0.3367 | Best: 0.0642
  Epoch  40 | Train: 0.2995 | Val: 0.2170 | Best: 0.0343
  Epoch  60 | Train: 0.2824 | Val: 0.0675 | Best: 0.0343
  Epoch  

In [15]:
def predict(test_data):
    models = {}
    for (n, k, m) in PARAM_COMBOS:
        mdl = MHeightDNN().to(device)
        mdl.load_state_dict(torch.load(model_path(n, k, m), map_location=device))
        mdl.eval()
        models[(n, k, m)] = mdl
    predictions = []
    for sample in test_data:
        n, k, m, P = sample
        X = torch.tensor(encode_sample(sample), dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            log2_pred = models[(int(n), int(k), int(m))](X).cpu().numpy()[0]
        predictions.append(float(np.maximum(2.0 ** log2_pred, 1.0)))
    return np.array(predictions)

#Results Explained:
The results below show 0.19 for a score which is roughly a 25% improvement. This is massive compared to the 3% that was found by optimizing the dropouts. Additional data was a key contributor in improving results.

In [16]:
def validation_prediction(train_data, train_heights):
    _, val_idx = train_test_split(np.arange(len(train_data)), test_size=0.1, random_state=42)
    val_samples = [train_data[i] for i in val_idx]
    val_true_heights = [train_heights[i] for i in val_idx]

    # encode with actual indices so LP feature is used for hard combos
    models = {}
    for (n, k, m) in PARAM_COMBOS:
        mdl = MHeightDNN().to(device)
        mdl.load_state_dict(torch.load(model_path(n, k, m), map_location=device))
        mdl.eval()
        models[(n, k, m)] = mdl

    val_predictions = []
    for i, sample in zip(val_idx, val_samples):
        n, k, m, P = sample
        X = torch.tensor(encode_sample(sample, sample_idx=i), dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            log2_pred = models[(int(n), int(k), int(m))](X).cpu().numpy()[0]
        val_predictions.append(float(np.maximum(2.0 ** log2_pred, 1.0)))

    val_predictions = np.array(val_predictions)

    for i in range(5):
        print(f'True: {val_true_heights[i]:.4f} | Predicted: {val_predictions[i]:.4f}')

    print(f'\nOverall average cost: {np.mean((np.log2(val_true_heights) - np.log2(val_predictions))**2):.4f}')

validation_prediction(train_data, train_heights)

True: 52.1556 | Predicted: 53.5196
True: 82.3333 | Predicted: 82.7830
True: 6396.3592 | Predicted: 7571.7129
True: 42.5556 | Predicted: 61.7411
True: 9599.3589 | Predicted: 10495.3242

Overall average cost: 0.0709


#Code Structure Explained:
The cells above handle data loading, feature engineering, and model training. The final cell below is self-contained and evaluates the pre-trained models without any retraining. All 9 trained model weights are included in the submission. A shared Google Drive link has been provided — if you authenticate and gain access, simply run the last cell as-is with no modifications needed.

In [18]:
import pickle
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from itertools import combinations
from scipy.optimize import linprog
from google.colab import drive
drive.mount('/content/drive')

#all of these files will be provided but you can gain access to the drive to make it easier as well

MODEL_DIR = '/content/drive/MyDrive/CSCE_636/project_2/models'
TRAIN_DATA_PATH = '/content/drive/MyDrive/CSCE_636/project_1/CSCE-636-Project-2-Train-n_k_m_P'
TRAIN_HEIGHT_PATH = '/content/drive/MyDrive/CSCE_636/project_1/CSCE-636-Project-2-Train-mHeights'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PARAM_COMBOS = [
    (9,4,2),(9,4,3),(9,4,4),(9,4,5),
    (9,5,2),(9,5,3),(9,5,4),
    (9,6,2),(9,6,3),
]

HARD_COMBOS = {(9,4,5), (9,5,4), (9,6,3)}
EXACT_MAX_COMBOS = {(9,4,5), (9,5,4), (9,6,3), (9,4,4), (9,5,3)}

# Load and merge both LP bound files
with open('/content/drive/MyDrive/CSCE_636/project_2/lp_bounds_hard_combos.pkl', 'rb') as f:
    lp_bounds = pickle.load(f)
with open('/content/drive/MyDrive/CSCE_636/project_2/lp_bounds_extra_combos.pkl', 'rb') as f:
    lp_bounds_extra = pickle.load(f)
lp_bounds.update(lp_bounds_extra)
print(f'Total LP bounds loaded: {len(lp_bounds):,}')

def compute_c11_features(n, k, P):
    r = n - k
    H = np.hstack([P.T, np.eye(r)])
    col_indices = list(range(n))
    values = []
    for S in combinations(col_indices, r):
        S = list(S)
        H_S = H[:, S]
        if abs(np.linalg.det(H_S)) < 1e-10:
            continue
        H_S_inv = np.linalg.inv(H_S)
        H_S_bar = H[:, [j for j in col_indices if j not in S]]
        for idx in range(len(S)):
            row = H_S_inv[idx, :]
            val = np.sum(np.abs(row @ H_S_bar))
            values.append(val)
    values = sorted(values, reverse=True)
    feat = np.zeros(10)
    feat[:min(len(values), 10)] = values[:10]
    return feat

def compute_lp_bound_online(n, k, m, P):
    G = np.hstack([np.eye(k), P])
    col_indices = list(range(n))
    best = 0.0
    for S in combinations(col_indices, m):
        S = list(S)
        S_bar = [j for j in col_indices if j not in S]
        G_Sbar = G[:, S_bar]
        A_ub = np.vstack([G_Sbar.T, -G_Sbar.T])
        b_ub = np.ones(2 * len(S_bar))
        for i in S:
            g_i = G[:, i]
            c = -g_i
            result = linprog(c, A_ub=A_ub, b_ub=b_ub,
                           bounds=[(None, None)] * k,
                           method='highs')
            if result.success:
                val = -result.fun
                if val > best:
                    best = val
    return best

def encode_sample(sample, sample_idx=None):
    n, k, m, P = sample
    col_norms = np.linalg.norm(P, axis=0)
    sort_idx  = np.argsort(-col_norms)
    P_sorted  = P[:, sort_idx]
    p_flat    = P_sorted.flatten()
    p_padded  = np.zeros(20)
    p_padded[:len(p_flat)] = p_flat
    p_padded  = p_padded / 30.0
    col_norms_padded = np.zeros(5)
    col_norms_padded[:len(col_norms)] = np.sort(col_norms)[::-1]
    col_norms_padded = col_norms_padded / 30.0
    row_norms = np.linalg.norm(P, axis=1)
    row_norms_padded = np.zeros(6)
    row_norms_padded[:len(row_norms)] = row_norms
    row_norms_padded = row_norms_padded / 30.0
    scalars = np.array([n/9.0, k/9.0, m/9.0, m/(n-k), k/n])
    combo = (int(n), int(k), int(m))
    if combo in HARD_COMBOS:
        c11_feats = compute_c11_features(n, k, P) / 10000.0
    else:
        c11_feats = np.zeros(10)
    if combo in EXACT_MAX_COMBOS:
        if sample_idx is not None and sample_idx in lp_bounds:
            lp_val = lp_bounds[sample_idx]
        else:
            lp_val = compute_lp_bound_online(int(n), int(k), int(m), P)
        lp_feat = np.array([np.log2(max(lp_val, 1.0)) / 25.0])
    else:
        lp_feat = np.array([0.0])
    return np.concatenate([scalars, p_padded, col_norms_padded, row_norms_padded, c11_feats, lp_feat]).astype(np.float32)

class MHeightDNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(47, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(1024, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def model_path(n, k, m):
    return f'{MODEL_DIR}/model_n{n}_k{k}_m{m}.pth'

def predict(test_data):
    models = {}
    for (n, k, m) in PARAM_COMBOS:
        mdl = MHeightDNN().to(device)
        mdl.load_state_dict(torch.load(model_path(n, k, m), map_location=device))
        mdl.eval()
        models[(n, k, m)] = mdl
    predictions = []
    for sample in test_data:
        n, k, m, P = sample
        X = torch.tensor(encode_sample(sample), dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            log2_pred = models[(int(n), int(k), int(m))](X).cpu().numpy()[0]
        predictions.append(float(np.maximum(2.0 ** log2_pred, 1.0)))
    return np.array(predictions)

with open(TRAIN_DATA_PATH, 'rb') as f:
    train_data = pickle.load(f)
with open(TRAIN_HEIGHT_PATH, 'rb') as f:
    train_heights = pickle.load(f)

_, val_idx = train_test_split(np.arange(len(train_data)), test_size=0.1, random_state=42)
val_samples = [train_data[i] for i in val_idx]
val_true_heights = [train_heights[i] for i in val_idx]

models = {}
for (n, k, m) in PARAM_COMBOS:
    mdl = MHeightDNN().to(device)
    mdl.load_state_dict(torch.load(model_path(n, k, m), map_location=device))
    mdl.eval()
    models[(n, k, m)] = mdl

val_predictions = []
for i, sample in zip(val_idx, val_samples):
    n, k, m, P = sample
    X = torch.tensor(encode_sample(sample, sample_idx=int(i)), dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        log2_pred = models[(int(n), int(k), int(m))](X).cpu().numpy()[0]
    val_predictions.append(float(np.maximum(2.0 ** log2_pred, 1.0)))

val_predictions = np.array(val_predictions)

for i in range(5):
    print(f'True: {val_true_heights[i]:.4f} | Predicted: {val_predictions[i]:.4f}')

print(f'\nOverall average cost: {np.mean((np.log2(val_true_heights) - np.log2(val_predictions))**2):.4f}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total LP bounds loaded: 389,972
True: 52.1556 | Predicted: 53.5196
True: 82.3333 | Predicted: 82.7830
True: 6396.3592 | Predicted: 7571.7129
True: 42.5556 | Predicted: 61.7411
True: 9599.3589 | Predicted: 10495.3242

Overall average cost: 0.0709


#Testing the model
This will be updated after the initial project submission

In [ ]:


# Load the test set
TEST_DATA_PATH = '/content/drive/MyDrive/CSCE_636/project_1/CSCE-636-Project-1-Test-n_k_m_P'

with open(TEST_DATA_PATH, 'rb') as f:
    test_data = pickle.load(f)

print(f'Test samples: {len(test_data)}')

# Run predictions (uses the models already loaded above)
test_predictions = predict(test_data)

# Save the output
OUTPUT_PATH = '/content/drive/MyDrive/CSCE_636/project_1/CSCE-636-Project-1-Test-mHeights'
with open(OUTPUT_PATH, 'wb') as f:
    pickle.dump(list(test_predictions), f)

print(f'Done! Saved {len(test_predictions)} predictions.')

#References
[1] R. M. Roth, Z. Zhu, C. Yuan, P. H. Siegel, and A. Jiang, "On the height profile of analog error-correcting codes," arXiv:2602.20366v1 [cs.IT], Feb. 2026.

[2] A. Jiang, "Analog error-correcting codes: Designs and analysis," IEEE Transactions on Information Theory, vol. 70, no. 11, pp. 7740-7756, Nov. 2024.
